# Needed imports

The following imports are needed for the Manager classes and the final main function.

In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy as db
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial.distance import cdist

***

# LogicManager Class

## Class Explanation

The LogicManager class is the core logic unit of the program. It encapsulates all logic necessary to evaluate, match, and validate relationships between training data, ideal functions, and test data based on the task’s requirements.

This class is separated from the database or visualization components to follow clean object-oriented principles and to keep a clear separation of concerns. It is designed to work on in-memory data structures, particularly pandas.DataFrame objects, and supports several core functionalities required by the task:

## Purpose and Functions

### get_best_fit_function(...)

This method identifies the best-fitting ideal function for a given training function.

 - It computes the least squares deviation between the training data and each of the 50 ideal functions.
 - The ideal function with the smallest sum of squared y-deviations is selected and its index returned.
 - This logic directly addresses part (i) of the assignment: choosing the ideal function with the best fit for each training function.

### calculate_max_deviation(...)

This function calculates the maximum Euclidean deviation between all training data points and the corresponding ideal function values.

 - It is used to define the tolerance boundary for mapping test points, following the assignment condition that deviations must not exceed the training deviation by more than √2.


### validate_deviation(...)

This method checks if a specific (x, y) test data point can be assigned to a given ideal function.

 - It locates the closest point on the ideal function and measures the Euclidean distance.
 - If the distance is within the allowed threshold, it returns the deviation; otherwise, it returns None.

### find_best_function_test(...)

This high-level method checks a test data point against all ideal functions selected during training.

 - It determines whether the point fits any of the ideal functions within the allowed deviation.
 - If multiple ideal functions are possible, the one with the smallest deviation is chosen.
 - This satisfies part (ii) of the assignment: matching each test point to a chosen ideal function if the mapping criterion is met.

In [ ]:
class LogicManager:
    def __init__(self) -> None:
        pass

    def get_best_fit_function(self, xy_train_func:pd.DataFrame, xy_all_ideal_func:pd.DataFrame):
        '''
        Find the best fitting ideal function for a train function

        :param xy_train_func: a train function
        :param xy_all_ideal_func: all possible ideal functions
        :return: index of best fitting function
        '''
        # Ensure x values match
        if not np.array_equal(xy_train_func.iloc[:, 0], xy_all_ideal_func.iloc[:, 0]):
            raise ValueError("X values in training and ideal datasets do not match")

        y_train = xy_train_func.iloc[:, 1].values
        best_function  = -1
        smalest_deviation = float('inf')
        for column in range(1, 51):
            y_ideal = xy_all_ideal_func.iloc[:, column].values
            # Least squares calculation
            deviation = np.sum((y_train - y_ideal) ** 2)  

            # Update with better function if smaller diviations has been found
            if deviation < smalest_deviation:
                smalest_deviation = deviation
                best_function = column

        return best_function

    def calculate_max_deviation(self, xy_train: np.array, xy_ideal: np.array) -> float:
        """
        Calculate the maximum point-wise Euclidean deviation between training data and ideal function.

        :param xy_train: Array of (x, y) coordinates of the training data
        :param xy_ideal: Array of (x, y) coordinates of the ideal function
        :return: Maximum deviation
        """
        # Ensure the input arrays are 2D
        xy_train = np.atleast_2d(xy_train)
        xy_ideal = np.atleast_2d(xy_ideal)

        # Calculate pairwise distances between all points
        distances = cdist(xy_train, xy_ideal)

        # For each training point, find the minimum distance to any ideal point
        min_distances = np.min(distances, axis=1)

        # Return the maximum of these minimum distances
        return np.max(min_distances)

    def validate_deviation(self, x_value, y_value, xy_func: pd.DataFrame, max_deviation):
        """
        Validate if the (x,y) coordiate fit into the max_diviation of the xy_func

        :param x_value: x value of coordinate
        :param y_value: y value of coordinate
        :param xy_func: function to validate with
        :param max_deviation: maximum deviation to function
        :return: If validatet the deviation of the coordinate, otherwise None
        """
        # Find the closest point on the curve
        distances = np.sqrt((xy_func.iloc[:, 0] - x_value)**2 + (xy_func.iloc[:, 1] - y_value)**2)
        closest_index = distances.idxmin()

        x_value_func = xy_func.iloc[closest_index, 0]
        y_value_func = xy_func.iloc[closest_index, 1]

        # Calculate the Euclidean distance
        deviation = np.sqrt((x_value - x_value_func)**2 + (y_value - y_value_func)**2)

        # Check if deviation doesn't exceed max_deviation
        if deviation <= max_deviation :
            return deviation

        # Coordinate could not be validated
        return None


    def find_best_function_test(self, x_value, y_value, dataFrame_ideal:pd.DataFrame, pd_func_max_div:pd.DataFrame):
        """
        Validate if the (x,y) coordiate fit into the max_diviation of the xy_func

        :param x_value: x value of coordinate
        :param y_value: y value of coordinate
        :param dataFrame_ideal: all ideal function
        :param pd_func_max_div: array with (choosen function, max deviation)
        :return: returns best deviation and the best fitting function
        """
        best_deviation = None
        best_function = None

        # Loop over all functions
        for index, row in pd_func_max_div.iterrows():
            func_id = row['func_id']
            max_div = row['max_div']

            deviation = self.validate_deviation(x_value, y_value, dataFrame_ideal.iloc[:, [0, func_id]], max_div)

            # Check if the result is the better option
            if deviation != None:
                if best_deviation == None or deviation < best_deviation:
                    best_deviation = deviation
                    best_function = func_id

        # Return solution
        return best_deviation, best_function



***

# VisualManager Class

## Class Explanation

The VisualManager class is responsible for creating a comprehensive and meaningful visualization of all data involved in the task — including training data, ideal functions, test data, and the deviations between them.

This class focuses purely on data visualization, separating visual concerns from database handling and logical processing (as done by the DatabaseManager and LogicManager classes). It makes use of the matplotlib library for rendering and is designed to support clarity, reproducibility, and visual insight into the program’s decisions.

## Purpose and Functions

### \_\_init__(...)

The constructor receives and stores the key datasets:

 - The training dataset
 - The ideal function dataset
 - The test dataset

These datasets are later used for plotting the relevant visual elements.

### visualize_data_and_deviations(...)

This is the main method that triggers the complete visualization.

 - It displays:
    - All four chosen ideal functions with their associated deviation zones
    - The training data points
    - he test data points, colored by whether and how they were matched
    - The unchosen ideal functions and unmatched test data in gray
    
 - The result is a single, comprehensive plot with legends, axis labels, and title, clearly indicating how the functions relate and how data was assigned.

### darken_color(...)

This helper function modifies a given color by darkening it (or optionally lightening it).
    
 - Used to visually distinguish matched test points from their corresponding ideal function curves in the plot.

### _plot_function_with_derivativ_area(...)

Plots an ideal function along with its deviation corridor — the allowed range around the function within which test points can still be matched.

 - The deviation area is calculated orthogonally to the function curve using vector mathematics and visualized as a shaded band.

### _plot_training_data(...)

Displays all training data points using gray scatter dots with low opacity.

 - Helps to visually indicate how densely the training data covers the function space.

### _plot_test_data(...)

Visualizes all test data points:

 - Matched test points are shown using slightly darkened versions of their corresponding function colors.
 - Unmatched test points are shown in gray.
 - Uses dynamic coloring and grouping based on assigned function IDs from the logic output.

### _plot_ideal_functions(...)

Plots all 50 ideal functions:

 - The four chosen functions are shown in full color with their deviation zones via _plot_function_with_derivativ_area(...).
 - The remaining 46 unchosen functions are plotted in light gray to provide context without clutter.

In [ ]:
class VisualManger:

    def __init__(self, df_train, df_ideal, df_test):
        """
        Saves localy need data fields

        :param df_train: train data
        :param df_ideal: ideal data
        :param df_test: test data
        """
        self.dataFrame_train = df_train
        self.dataFrame_ideal = df_ideal
        self.dataFrame_test = df_test

    def visualize_data_and_deviations(self, func_x_max_dev:pd.DataFrame, function_colors):
        """
        Visualisation of the chosen and unchosen functions, all train data, all deviation zones 
        and test data with its matched function color (if it matched)

        :param func_x_max_dev: the chosen functions matched with there individual deviation
        :param function_colors: color for the functions, last color for unmatchend test data and unchosen functions
        """

        # Plot aspect ratio
        plt.figure(figsize=(15, 10))
    
        # Plot ideal functions
        self._plot_ideal_functions(func_x_max_dev, function_colors)
    
        # Plot training data
        self._plot_training_data()
    
        # Plot test data
        self._plot_test_data(func_x_max_dev['func_id'], function_colors)
    
        # Visualize everything
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.title('Data Visualization with Deviations')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.xlim(-20, 20)
        plt.ylim(-20, 20)
        plt.tight_layout()
        plt.show()

    def darken_color(self, hex_color, factor):
        """
        Darken the color by muliplying it with a factor less than 1 (could also 
        be used to enlight the color with a multiplier higher than 1)

        :param hex_color: the color to manipulate
        :param factor: multiplier
        :return: darker color
        """

        # Convert hex to rgb
        rgb = mcolors.hex2color(hex_color)
        
        # Darken rgb by multiplying with factor
        darkened_rgb = tuple([channel * factor for channel in rgb])
        
        # Convert hex back to rgb
        darkened_hex = mcolors.to_hex(darkened_rgb)
        
        return darkened_hex
    

    def _plot_function_with_derivativ_area(self, x,y, max_deviation, function_color, text):
        """
        Plotting a function by x and y coordinates + an deviation area around it set by max_deviation

        :param x: x values of the function
        :param y: y values of the function
        :param max_deviation: size of deviation area around the function (for one side)
        :param function_color: display color of the function and its deviation area
        :param text: text to the explanation fiel on the side
        """

        # Calculate the derivative
        y_derivative = np.gradient(y, x)
    
        # Width of the orthogonal zone
        z = max_deviation
    
        # Calculate the normal vectors
        normal_x = -y_derivative / np.sqrt(1 + y_derivative**2)
        normal_y = 1 / np.sqrt(1 + y_derivative**2)
    
        # Calculate the upper and lower bounds
        x_upper = x + z * normal_x
        y_upper = y + z * normal_y
    
        x_lower = x - z * normal_x
        y_lower = y - z * normal_y

    
        # Create the contour of the zone
        x_fill = np.concatenate([x_upper, x_lower[::-1]])
        y_fill = np.concatenate([y_upper, y_lower[::-1]])
    
        plt.plot(x, y, label=text, color= function_color, linewidth=2)
        plt.fill(x_fill, y_fill, color= function_color, alpha=0.4)
    
    def _plot_training_data(self):
        """
        Scatters all the trainings data in gray and low alpha
        """
        # Boolean helper variable to only print the label one time
        label_printed = False
        # Go though the whole dataFrame_train and scatter each dot in gray 
        for col in self.dataFrame_train.columns[1:]:  # Assuming first column is 'X'
            if label_printed == True:
                plt.scatter(self.dataFrame_train['X'], self.dataFrame_train[col], alpha=0.2, s=20, c='gray')
            else:    
                plt.scatter(self.dataFrame_train['X'], self.dataFrame_train[col], alpha=0.2, label=f'Training data', s=20, c='gray')
                label_printed = True

    
    def _plot_test_data(self, chosen_functions, function_colors):
        """
        Scatter the matched and unmatched test data from the chosen functions, unmatched test data will be displayed in gray

        :param chosen_functions: the chosen function from the ideal function data set in order of the function_colors param
        :param function_colors: the function colors in order of the chosen_function param
        """

        # 5 lists for (x, y) pairs
        array_xy = [[] for _ in range(len(chosen_functions)+1)]  
    
        for _, row in self.dataFrame_test.iterrows():
            x, y = row['X (test func)'], row['Y (test func)']
            func_num = row['No. of ideal func']
            
            # Filter for chosen and unchosen test data
            if pd.notna(func_num):
                if func_num in chosen_functions.values:
                    index = chosen_functions[chosen_functions == func_num].index[0]
                    array_xy[index].append((x,y))         
            else:
                array_xy[4].append((x,y))  
                
        # Scatter matched test data
        for index in range(len(array_xy)-1):
            plt.scatter([pair[0] for pair in array_xy[index]], 
                        [pair[1] for pair in array_xy[index]], 
                        color= self.darken_color(function_colors[index],0.95), 
                        label='Matched Test Data', 
                        s=30)
        
        # Scatter unmatched test data
        plt.scatter([pair[0] for pair in array_xy[len(array_xy)-1]], 
                    [pair[1] for pair in array_xy[len(array_xy)-1]], 
                    color='gray', 
                    label='Unmatched Test Data', 
                    s=30)
    
    def _plot_ideal_functions(self, func_x_max_dev:pd.DataFrame, function_colors):
        """
        Scatter the matched and unmatched test data from the chosen functions, unmatched test data will be displayed in gray

        :param func_x_max_dev: the chosen functions matched with there individual deviation
        :param function_colors: the function colors in order of the chosen function from func_x_max_dev param
        """
        # Plot ideal functions
        x = self.dataFrame_ideal['X']
        for i in range(1, len(self.dataFrame_ideal.columns)):  # Start from column index 1
            y = self.dataFrame_ideal.iloc[np.searchsorted(self.dataFrame_ideal['X'], x), i]
            
            # Find the chosen functions
            if i in func_x_max_dev['func_id'].values:
                index = func_x_max_dev[func_x_max_dev['func_id'] == i].index[0]
                self._plot_function_with_derivativ_area(x,y,
                                                        func_x_max_dev['max_div'].at[index], 
                                                        function_colors[index], 
                                                        f'Chosen function {i}')
            # Unchosen function are displayed in gray
            else:
                plt.plot(x, y, color='gray', linewidth=1, alpha=0.2)

***

# DatabaseManager Class

## Class Explanation

The DatabaseManager class is responsible for handling all database-related operations in the program. Its main purpose is to ensure that training data, ideal functions, and test results are correctly read from CSV files, stored in a SQLite database, and later retrieved when needed for processing or visualization.

This class acts as the data backbone of the entire solution. It uses the SQLAlchemy library to interface with a relational SQLite database and pandas for data manipulation. By centralizing all database logic into one place, it ensures maintainability, reusability, and separation of concerns.

## Purpose and Functions

### \_\_init__(...)

Initializes the database engine using the provided path to the SQLite database file.

 - This engine is reused across all operations to interact with the database.

### createDatabase(...)

Creates all required tables (train_db, ideal_db, and test_db) in the database if they do not already exist.

 - Each table follows the schema provided in the assignment.
 - Column naming is aligned exactly with the specifications (e.g., "Y1 (training func)", "No. of ideal func", etc.).

### csv_2DArray(...)

Helper method that loads a CSV file from a given path into a pandas.DataFrame.

### import_trainCSV(...)

Imports training data from a CSV file into the train_db table.

 - Iterates over each row and calls trainDB_add_record(...) to insert data.
 - Returns the number of successfully inserted rows.

### import_idealCSV(...)

Imports ideal function data from a CSV file into the ideal_db table.

 - Handles the broader structure with 50 Y columns.
 - Each row is passed to idealDB_add_record(...).

### trainDB_add_record(...)

Inserts a single row into the train_db table using parameterized SQL statements.

 - Returns True if the insertion succeeds, or False if an exception occurs.

### idealDB_add_record(...)

Inserts a single row into the ideal_db table with 50 ideal function values.

 - Dynamically builds the SQL query and parameter list for all 50 columns.

### testDB_add_record(...)

Inserts a single test data result into the test_db table, including the x/y values, the calculated deviation, and the matched ideal function number.

 - Used after the logic class has completed its analysis on the test dataset.

### load_table(...)

Loads an entire table (any of the three database tables) into a pandas.DataFrame.

 - Provides easy access to database content for downstream tasks like logic evaluation or visualization.

In [ ]:
class DatabaseManager:
    def __init__(self, db_path):
        '''
        Creats/Loads database engine

        :param db_path: path to the database
        '''
        # Create engine so it can be used in the whole class
        self.db_engine = db.create_engine(f'sqlite:///{db_path}')

    def load_table(self, table_name):
        '''
        Loads table into a pandas data frame

        :param table_name: name of the table to load
        :return: panda data frame of table
        '''

        table = db.Table(table_name, db.MetaData(), autoload_with=self.db_engine)
        select_statement = db.select(table)
        # Connect to database and get the table
        with self.db_engine.connect() as connection:
            result = connection.execute(select_statement)

            # Fetch all results into a list of tuples
            rows = result.fetchall()

            # Convert table into a dataframe
            column_names = table.columns.keys()
            df = pd.DataFrame(rows, columns=column_names)

            return df

    def csv_2DArray(self, directory):
        '''
        Read csv file into a pandas data frame

        :param directory: directory of csv file
        :return: panda data frame of csv file
        '''
        return pd.read_csv(directory)

    def import_trainCSV(self, directory):
        '''
        Import the train data from the train.csv into the database
        
        :param directory: directory of csv file
        :return: size of successfull added records
        '''
        counter = 0
        train_df = self.csv_2DArray(directory)
        for ind in train_df.index:
            if self.trainDB_add_record(train_df['x'][ind], train_df['y1'][ind], train_df['y2'][ind], train_df['y3'][ind], train_df['y4'][ind]): 
                counter += 1
        
        # Return the amount of records that has been added
        return counter

    def import_idealCSV(self, directory):
        '''
        Import the ideal data from the ideal.csv into the database

        :param directory: directory of csv file
        :return: size of successfull added records
        '''
        counter = 0
        ideal_df = self.csv_2DArray(directory)
        for ind in ideal_df.index:
            x_value = ideal_df['x'][ind]
            y_values = ideal_df.loc[ind, 'y1':'y50'].values
            # on success increase the counter by one
            if self.idealDB_add_record(x_value, y_values): 
                counter += 1
        
        # Return the amount of records that has been added
        return counter


    def trainDB_add_record(self, x, y1, y2, y3, y4):
        '''
        Add a record to the train table in the database

        :param x: X value
        :param y1: Y1 (training func) value
        :param y2: Y2 (training func) value
        :param y3: Y3 (training func) value
        :param y4: Y4 (training func) value
        :param directory: directory of csv file
        :return: BOOL if successfull
        '''
        connection = self.db_engine.connect()
        try:
            # Creation SQL statement with placeholder
            sql = db.text("""
                INSERT INTO train_db 
                (`X`, `Y1 (training func)`, `Y2 (training func)`, `Y3 (training func)`, `Y4 (training func)`)
                VALUES (:x, :y1, :y2, :y3, :y4)
            """)

            # Parameter with input data
            params = {
                'x': x,
                'y1': y1,
                'y2': y2,
                'y3': y3,
                'y4': y4
            }

            # Execute SQL statement
            connection.execute(sql, params)
            connection.commit()
            return True

        except Exception as e:
            if "Duplicate entry" in str(e) or "UNIQUE constraint failed" in str(e):
                print(f"Record with X={x} already exists in train_db, skipping insert.")
                connection.rollback()
                return False
            else:
                print(f"Error while INSERT operation in train_db: {e}")
                connection.rollback()
                return False

        finally:
            # Close connection
            connection.close()

       
        
    def idealDB_add_record(self, x, y_values):
        '''
        Add a record to the train table in the database

        :param x: X value
        :param y_values: array containing all y indexes from 1 to 50
        :param directory: directory of csv file
        :return: BOOL if successfull
        '''
        connection = self.db_engine.connect()
        try:
            # Create column name string for SQL statement
            columns = ['`X`'] + [f'`Y{i} (ideal func)`' for i in range(1, 51)]
            column_string = ', '.join(columns)

            # Create value placeholder
            value_placeholders = [':x'] + [f':y{i}' for i in range(1, 51)]
            value_string = ', '.join(value_placeholders)

            # Creation of SQL statement with placeholder
            sql = f"""INSERT INTO ideal_db ({column_string}) VALUES ({value_string})"""

            # Parameter with input data
            params = {'x': x}
            params.update({f'y{i+1}': y for i, y in enumerate(y_values)})

            # Execute SQL statement
            connection.execute(db.text(sql), params)
            connection.commit()
            return True

        except Exception as e:
            if "Duplicate entry" in str(e) or "UNIQUE constraint failed" in str(e):
                print(f"Record with X={x} already exists in ideal_db, skipping insert.")
                connection.rollback()
                return False
            else:
                print(f"Error while INSERT operation in ideal_db: {e}")
                connection.rollback()
                return False

        finally:
            # Close connections
            connection.close()

    
    def testDB_add_record(self, x_test, y_test, delta_y_test, no_ideal_func):
        '''
        Add a record to the test table in the database

        :param x_test: X value
        :param y_test: Y (test func) value
        :param delta_y_test: Delta Y (test func) value
        :param no_ideal_func: No. of ideal func value
        :return: BOOL if successful or skipped due to duplicate PK
        '''
        connection = self.db_engine.connect()
        try:
            # Creation of SQL statement with placeholder
            sql = db.text("""
                INSERT INTO test_db 
                (`X (test func)`, `Y (test func)`, `Delta Y (test func)`, `No. of ideal func`)
                VALUES (:x_test, :y_test, :delta_y_test, :no_ideal_func)
            """)

            # Parameter with input data
            params = {
                'x_test': x_test,
                'y_test': y_test,
                'delta_y_test': delta_y_test,
                'no_ideal_func': no_ideal_func
            }

            # Execute SQL statement
            connection.execute(sql, params)
            connection.commit()
            return True

        except Exception as e:
            if "Duplicate entry" in str(e) or "UNIQUE constraint failed" in str(e):
                print(f"Record with X={x_test} already exists in test_db, skipping insert.")
                connection.rollback()
                return False
            else:
                print(f"Error while INSERT operation in test_db: {e}")
                connection.rollback()
                return False

        finally:
            # Close connections
            connection.close()
    

    def createDatabase(self):
        '''
        Creates all needed database tabels at the choosen direction, if not already exist

        :return: true if successfull
        '''
        # Get connection object
        connection = self.db_engine.connect()

        try:
            # Get meta data object
            meta_data = db.MetaData()

            # Create train_db table Y1-Y4
            y_columns = [
                db.Column(f'Y{i} (training func)', db.DOUBLE_PRECISION) for i in range(1, 5)
            ]
            
            # Combine the X coulumn with the Y1 to Y4 columns
            train_db = db.Table(
                'train_db',
                meta_data,
                db.Column('X', db.DOUBLE_PRECISION, primary_key=True, autoincrement=False),
                *y_columns,
            )

            # Create ideal_db table Y1-Y50
            y_columns = [
                db.Column(f'Y{i} (ideal func)', db.DOUBLE_PRECISION) for i in range(1, 51)
            ]

            # Combine the X coulumn with the Y1 to Y50 columns
            ideal_db = db.Table(
                'ideal_db',
                meta_data,
                db.Column('X', db.DOUBLE_PRECISION, primary_key=True, autoincrement=False, nullable=True,),
                *y_columns,
            )

            # Create test_db table
            test_db = db.Table(
                'test_db',
                meta_data,
                db.Column('X (test func)', db.DOUBLE_PRECISION, primary_key=True, autoincrement=False, nullable=True),
                db.Column('Y (test func)', db.DOUBLE_PRECISION),
                db.Column('Delta Y (test func)', db.DOUBLE_PRECISION),
                db.Column('No. of ideal func', db.DOUBLE_PRECISION)
            )

            # Create train_db table and stores the information in metadata
            meta_data.create_all(self.db_engine)

            # On success return True
            return True

        except Exception as e:
            print(f"Error while database creation: {e}")
            connection.rollback()
            return False

        finally:
            # Close connection
            connection.close()

# Main Function

The main() function orchestrates the entire workflow of the program — from data import to logic processing and final visualization. It integrates all components (DatabaseManager, LogicManager, VisualManager) to solve the assignment task end-to-end.

In [ ]:
def main():
    # ------------------------------DATABASE-------------------------------- #

    # Create Database
    db_manager = DatabaseManager("dataBase.db")
    db_manager.createDatabase()

    # Import ideal CSV into database
    db_manager.import_idealCSV("./data/ideal.csv")

    # Import train CSV into database
    db_manager.import_trainCSV("./data/train.csv")

    # Load (just created) train and ideal table
    dataFrame_ideal = db_manager.load_table("ideal_db")
    dataFrame_train = db_manager.load_table("train_db")

    # Load test CSV into pandas data frame
    csv_test = db_manager.csv_2DArray("./data/test.csv")


    # ------------------------------LOGIC----------------------------------- #
    # Create logic manager
    lgc_manager = LogicManager()

    # Find the four best fitting functions
    # Function 1
    ideal_for_y1 = lgc_manager.get_best_fit_function(
        dataFrame_train.iloc[:,[0,1]], 
        dataFrame_ideal)
    max_diviation_y1 = lgc_manager.calculate_max_deviation(
        dataFrame_train.iloc[:, [0,1]], 
        dataFrame_ideal.iloc[:, [0,ideal_for_y1]])

    # Function 2
    ideal_for_y2 = lgc_manager.get_best_fit_function(
        dataFrame_train.iloc[:,[0,2]], 
        dataFrame_ideal)
    max_diviation_y2 = lgc_manager.calculate_max_deviation(
        dataFrame_train.iloc[:, [0,2]], 
        dataFrame_ideal.iloc[:, [0, ideal_for_y2]])

    # Function 3
    ideal_for_y3 = lgc_manager.get_best_fit_function(
        dataFrame_train.iloc[:,[0,3]], 
        dataFrame_ideal)
    max_diviation_y3 = lgc_manager.calculate_max_deviation(
        dataFrame_train.iloc[:, [0,3]], 
        dataFrame_ideal.iloc[:, [0, ideal_for_y3]])

    # Function 4
    ideal_for_y4 = lgc_manager.get_best_fit_function(
        dataFrame_train.iloc[:,[0,4]], 
        dataFrame_ideal)
    max_diviation_y4 = lgc_manager.calculate_max_deviation(
        dataFrame_train.iloc[:, [0,4]], 
        dataFrame_ideal.iloc[:, [0, ideal_for_y4]])

    # Convert to usabel pandas data frame
    # "* np.sqrt(2)" for the max diviation of the test data
    pd_func_max_div = pd.DataFrame([
        [ideal_for_y1, max_diviation_y1 * np.sqrt(2)],
        [ideal_for_y2, max_diviation_y2 * np.sqrt(2)], 
        [ideal_for_y3, max_diviation_y3 * np.sqrt(2)],
        [ideal_for_y4, max_diviation_y4 * np.sqrt(2)]],
        columns=['func_id', 'max_div'])

    for index, row in csv_test.iterrows():
        x_value = csv_test.iloc[index, 0] 
        y_value = csv_test.iloc[index, 1] 

        # Find the beste function and its deviation for the test table
        deviation_and_funcID = lgc_manager.find_best_function_test(
            x_value, y_value, dataFrame_ideal, pd_func_max_div)

        # Import result into the database  
        db_manager.testDB_add_record(
            x_value, y_value, deviation_and_funcID[0], deviation_and_funcID[1]);        


    # ---------------------------VISUALISATION------------------------------- #
    # Load the test database for visualisation
    dataFrame_test = db_manager.load_table("test_db")

    # Colors for each of the four functions
    function_colors = ['#f56fa1', 
                       '#f0de89',
                       '#90d2d8',
                       '#63bc46']

    # Create VisualManager
    v_manager = VisualManger(
        dataFrame_train, dataFrame_ideal, dataFrame_test)
    # Start visualisation procedure
    v_manager.visualize_data_and_deviations(pd_func_max_div , function_colors)

# Final Execution

The final data visualisation is shown after the execution of the main function.

In [ ]:
main()

#### 